# Rolling Walk-Forward Approach

As discussed in [notebook 2](02_baseline_cointegration.ipynb), the absence of full-history cointegration is consistent with the relationship being unstable across regimes. To check for localised short-term relationships we may employ a rolling approach. This means we test for cointegration in lookback windows of $n$ days in a rolling manner. We may use a smaller step size $s$ than our lookback window size $n$. This will give us overlapping windows to ensure that a regime that starts or ends somewhere in the middle of one of our blocks doesn't go undetected. However, it is important to note that a smaller $s$ increases the overlap between consecutive windows, so results across neighbouring windows should not be read as independent.

If we find any periods of cointegration, we can then fit a regression model and walk forward $m$ days to test whether the relationship holds. Note that testing whether the fitted model has stationary residuals here does not suffer from the same bias issue as if we were testing it on the data it had been fitted on, so we may employ an ADF test rather than an EG test here.

We treat these as two separate questions: whether short-term cointegrating relationships occur at all (the rolling lookback search), and whether relationships identified this way continue to hold out-of-sample (addressed separately once candidate windows have been identified).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint

In [2]:
df = pd.read_csv("../data/pep_ko_daily.csv", index_col=0)

## Rolling Lookback Window Length and Step Size 

The length of the rolling window and step size ($n$ and $s$ respectively) are parameters to be experimented with. 

We will start with $n=100$ (roughly 20 weeks, dependent on holidays) and $s=25$. Admittedly, this is a rather arbitrary starting choice aimed at giving ourselves enough data to fit the model whilst minimising our exposure to the same regime-breaking issues as [notebook 2](02_baseline_cointegration.ipynb) and also maximising our chances of finding regimes.

With this in mind, it is important that we properly correct our $5\%$ significance threshold to account for the number of tests we will be running. We will do this using a simple Bonferroni correction. However, note that since we are conducting many tests that are not independent, not least due to the overlapping windows, this will be a conservative adjustment. 

In [4]:
N_LOOKBACK = 100
STEP_SIZE = 25

SIG_LEVEL = 0.05

In [16]:
num_tests = int(1 + np.floor((len(df) - N_LOOKBACK) / STEP_SIZE))
sig_level_adj_regimes = SIG_LEVEL / num_tests

print("Number of individual tests ran at this stage: ", num_tests)
print("Adjusted per-test significance level for regime identification: ", sig_level_adj_regimes)

Number of individual tests ran at this stage:  544
Adjusted per-test significance level for regime identification:  9.191176470588235e-05


## Regime Identification